# Tau Complexity Test — press via oppgåvekompleksitet

**Lærdomar frå tau_stress_test:**
- r_max = min(N, d) er feil — voks med kontekst og drog τ ned automatisk
- Kontekstlengde åleine er ikkje kognitiv press

**Fiks:** r_max = hidden_dim (768 for GPT-2) — konstant normalisering

**Ny stressvariabel:** oppgåvekompleksitet — same lengde, stigande vanskegrad

**Hypotese:** τ stig når modellen møter genuint krevjande innhald
(matematikk, logikk, motstridande krav) samanlikna med enkel tekst

In [ ]:
!pip install transformers torch matplotlib numpy -q

In [ ]:
import torch, math, numpy as np, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

EULER_MASCHERONI = 0.5772156649015328
APERY            = 1.2020569031595942
TAU_MIN = math.exp(-EULER_MASCHERONI)  # 0.5615
TAU_MAX = 1.0 / APERY                  # 0.8319

print(f"Goldilocks: [{TAU_MIN:.4f}, {TAU_MAX:.4f}]")

def compute_tau(hidden_state, r_max_override=None):
    """r_max_override: bruk hidden_dim i staden for min(N,d)"""
    H = hidden_state.float()
    N, d = H.shape
    # FIKS: alltid normaliser mot hidden_dim, ikkje mot seq_len
    r_max = r_max_override if r_max_override else d
    S = torch.linalg.svdvals(H)
    S_sq = S ** 2
    p = S_sq / (S_sq.sum() + 1e-12)
    H_entropy = -(p * torch.log(p + 1e-12)).sum().item()
    r_eff = math.exp(H_entropy)
    tau   = r_eff / r_max
    return {"tau": tau, "r_eff": r_eff, "r_max": r_max, "H": H_entropy,
            "goldilocks": TAU_MIN <= tau <= TAU_MAX}

In [ ]:
MODEL_NAME = "gpt2"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
mod = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True)
mod.eval()
HIDDEN_DIM = mod.config.n_embd  # 768 for GPT-2
print(f"Lasta: {MODEL_NAME} — hidden_dim={HIDDEN_DIM}")

In [ ]:
# Stigande kompleksitet — alle ~100 tokens
# Frå enkel gjentaking til genuint krevjande resonnement

PROMPTS = {
    "1_repetitivt":    "the the the the the the the the the the the the the the the the "
                       "the the the the the the the the the the the the the the the the "
                       "the the the the the the the the the the the the the the the the "
                       "the the the the the the the the the the the the the the the the",

    "2_enkel_prosa":   "The cat sat on the mat. The dog ran in the park. The bird flew "
                       "over the tree. The fish swam in the lake. The sun rose in the east. "
                       "The moon set in the west. The wind blew through the leaves. "
                       "The rain fell on the ground. The snow covered the hills.",

    "3_narrativ":      "In 1687 Isaac Newton published his Principia Mathematica, "
                       "establishing the laws of motion and universal gravitation. "
                       "This work unified terrestrial and celestial mechanics under "
                       "a single mathematical framework, showing that the same force "
                       "governing falling apples also governs planetary orbits. "
                       "The implications transformed natural philosophy into physics.",

    "4_matematikk":    "Let f(x) = x^3 - 3x + 2. Find all real roots. "
                       "Note that f(1) = 0, so (x-1) is a factor. "
                       "Dividing: x^3 - 3x + 2 = (x-1)(x^2+x-2) = (x-1)(x+2)(x-1) = (x-1)^2(x+2). "
                       "Roots: x=1 (double) and x=-2. "
                       "Verify: f(-2) = -8+6+2 = 0. Confirmed. "
                       "Now consider g(x) = f'(x) = 3x^2-3 = 3(x-1)(x+1). "
                       "Critical points at x=1 and x=-1.",

    "5_logikk":        "If all ravens are black, and this is a raven, then this is black. "
                       "But consider: observing a green apple confirms 'all non-black things "
                       "are non-ravens' which is logically equivalent to 'all ravens are black'. "
                       "This is Hempel's paradox of confirmation. The paradox arises because "
                       "inductive logic applies symmetrically across contrapositive statements, "
                       "yet our intuitions about evidence are not symmetric in this way.",

    "6_motsetning":    "Task: simultaneously maximize and minimize the following function. "
                       "Constraint A requires x > 5. Constraint B requires x < 3. "
                       "Constraint C requires x = 4. All three constraints must hold. "
                       "Additionally, the function must be both continuous and discontinuous "
                       "at x=4. Resolve this while maintaining formal logical consistency "
                       "and provide a constructive proof of your solution.",
}

for name, text in PROMPTS.items():
    n_tok = tok(text, return_tensors="pt")["input_ids"].shape[1]
    print(f"{name}: {n_tok} tokens")

In [ ]:
# HOVUDTEST: tau per kompleksitetsnivå
print(f"\n{'Prompt':<22} {'tau':<10} {'r_eff':<10} {'H':<10} Status")
print("-" * 65)

complexity_results = {}

for name, text in PROMPTS.items():
    tokens = tok(text, return_tensors="pt", truncation=True, max_length=512)["input_ids"]
    with torch.no_grad():
        out = mod(input_ids=tokens)
    last_hidden = out.hidden_states[-1][0]  # [seq_len, hidden_dim]
    r = compute_tau(last_hidden, r_max_override=HIDDEN_DIM)
    complexity_results[name] = r
    status = "GOLDILOCKS ✓" if r["goldilocks"] else ("OVER ▲" if r["tau"] > TAU_MAX else "UNDER ▼")
    print(f"{name:<22} {r['tau']:<10.4f} {r['r_eff']:<10.2f} {r['H']:<10.3f} {status}")

print(f"\nGoldilocks: [{TAU_MIN:.4f}, {TAU_MAX:.4f}]")

In [ ]:
# TILLEGG: tau per lag for alle kompleksitetsnivå
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = plt.cm.viridis(np.linspace(0, 1, len(PROMPTS)))

# Venstre: tau per lag
ax = axes[0]
for i, (name, text) in enumerate(PROMPTS.items()):
    tokens = tok(text, return_tensors="pt", truncation=True, max_length=512)["input_ids"]
    with torch.no_grad():
        out = mod(input_ids=tokens)
    layer_taus = [compute_tau(hs[0], r_max_override=HIDDEN_DIM)["tau"] for hs in out.hidden_states]
    ax.plot(layer_taus, marker='o', markersize=3, linewidth=1.5,
            color=colors[i], label=name.split('_', 1)[1])

ax.axhspan(TAU_MIN, TAU_MAX, alpha=0.10, color='green')
ax.axhline(TAU_MIN, color='green', linestyle='--', linewidth=1)
ax.axhline(TAU_MAX, color='red',   linestyle='--', linewidth=1)
ax.set_xlabel("Lag"); ax.set_ylabel("τ")
ax.set_title("τ per lag — alle kompleksitetsnivå")
ax.set_ylim(0, 1.0); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Høgre: tau siste lag vs kompleksitet
ax2 = axes[1]
names = list(complexity_results.keys())
taus  = [complexity_results[n]["tau"] for n in names]
labels = [n.split('_', 1)[1] for n in names]
bar_colors = ['red' if t < TAU_MIN else ('orange' if t > TAU_MAX else 'green') for t in taus]
ax2.bar(range(len(names)), taus, color=bar_colors, alpha=0.8, edgecolor='white')
ax2.axhspan(TAU_MIN, TAU_MAX, alpha=0.10, color='green')
ax2.axhline(TAU_MIN, color='green', linestyle='--', linewidth=1, label=f'τ_min={TAU_MIN:.4f}')
ax2.axhline(TAU_MAX, color='red',   linestyle='--', linewidth=1, label=f'τ_max={TAU_MAX:.4f}')
ax2.set_xticks(range(len(names)))
ax2.set_xticklabels(labels, rotation=20, ha='right', fontsize=9)
ax2.set_ylabel("τ (siste lag)")
ax2.set_title("τ vs kompleksitet — siste lag")
ax2.set_ylim(0, 1.0); ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle(f"Tau Complexity Test — {MODEL_NAME} (r_max=hidden_dim={HIDDEN_DIM})",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"tau_complexity_{MODEL_NAME}.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Lagra: tau_complexity_{MODEL_NAME}.png")

In [ ]:
print("=" * 60)
print("HYPOTESE-TEST: tau stig med kompleksitet")
print("=" * 60)

names = list(complexity_results.keys())
taus  = [complexity_results[n]["tau"] for n in names]

tau_enkel   = taus[0]   # repetitivt
tau_kompleks = taus[-1]  # motsetning
delta = tau_kompleks - tau_enkel

print(f"\nEnklaste (repetitivt):  τ = {tau_enkel:.4f}")
print(f"Komplekse (motsetning): τ = {tau_kompleks:.4f}")
print(f"Endring:                {delta:+.4f}")

trend = all(taus[i] <= taus[i+1] + 0.01 for i in range(len(taus)-1))
print(f"\nMonoton stiging:        {'JA (grovt)' if trend else 'NEI'}")
goldilocks_naadd = any(TAU_MIN <= t <= TAU_MAX for t in taus)
print(f"Goldilocks nådd:        {'JA ✓' if goldilocks_naadd else 'NEI ✗'}")

print("\n" + "=" * 60)
if delta > 0.01:
    print("Konklusjon: τ stig med kompleksitet. KONSISTENT med hypotesen.")
elif abs(delta) < 0.01:
    print("Konklusjon: τ flat. Kompleksitet påverkar ikkje τ i GPT-2.")
else:
    print("Konklusjon: τ fell med kompleksitet. MOTSEIER hypotesen.")
print("=" * 60)

## Tolkingsguide

| Resultat | Tyding |
|---|---|
| τ stig mot Goldilocks med kompleksitet | Hypotesen bekrefta — kognitiv press er variabelen |
| τ flat uavhengig av oppgåve | GPT-2 for liten, treng 7B+ |
| τ fell med kompleksitet | Høg kompleksitet gir meir spreiing — motseier hypotesen |

**Neste: Mistral-7B eller Qwen2.5-7B** — der forventer vi τ ≈ 0.26 ved enkel tekst, høgare ved kompleks